# 00_env_config

Environment bootstrap for FabricOps Starter Kit notebooks.
This notebook defines environment-wide values and assembles framework config.
Reusable functions come from `fabricops_kit` package modules.


In [ ]:
# Import key functions needed by the environment bootstrap.
# Make sure the Fabric environment already has fabricops installed as a custom library.

from fabricops_kit import setup_metadata_tables
from fabricops_kit.fabric_input_output import (
    FabricStore,
    read_lakehouse_csv,
    read_lakehouse_table,
    write_lakehouse_table,
    read_warehouse_table,
    write_warehouse_table,
)
from fabricops_kit.config import (
    AIPromptConfig,
    FrameworkConfig,
    NotebookRuntimeConfig,
    PathConfig,
    DataAgreementConfig,
    _validate_audit_timezone,
    setup_notebook,
)

In [ ]:
print("FabricOps Starter Kit environment configuration loaded.")


# List of configs

### Runtime Config

In [ ]:
# Change this if you want other prefixes for setup_notebook naming checks.
NOTEBOOK_PREFIXES = ("00_env_config", "01_agreement", "02_pipeline", "03_governance", "99_explore")

RUNTIME_CONFIG = NotebookRuntimeConfig(NOTEBOOK_PREFIXES)

# FabricOps audit and technical timestamp timezone.
# UTC is the default and recommended portable option.
# To use local audit time, set a valid IANA timezone such as:
# "Asia/Singapore", "Australia/Sydney", or "America/New_York".
FABRICOPS_AUDIT_TIMEZONE = "Asia/Singapore"
_validate_audit_timezone(FABRICOPS_AUDIT_TIMEZONE)

# Lakehouse schema routing. Set LAKEHOUSE_SCHEMAS_ENABLED = False for classic/non-schema Lakehouses.
# Use None for classic/non-schema Lakehouses.
# Use configured schema names such as "dbo" for schema-enabled Lakehouses.
LAKEHOUSE_SCHEMAS_ENABLED = True
DEFAULT_LAKEHOUSE_SCHEMA = "dbo"
SOURCE_SCHEMA = DEFAULT_LAKEHOUSE_SCHEMA
UNIFIED_SCHEMA = DEFAULT_LAKEHOUSE_SCHEMA
PRODUCT_SCHEMA = DEFAULT_LAKEHOUSE_SCHEMA
METADATA_SCHEMA = "METADATA"

# Use "warn" during setup. Use "strict" when you want missing prerequisites to fail the bootstrap.
VALIDATION_MODE = "warn"


### Path Config

In [ ]:
# Change this if needed for your own custom-defined environments, for example: dev, qat, prd.
ENV = "dev"
ENV_NAME = ENV

REQUIRED_TARGETS = ["source", "unified", "product", "metadata"]

# Target names should align with REQUIRED_TARGETS above.
ENV_PATHS = {
    ENV: {
        "source": FabricStore(
            env=ENV,
            workspace_id="68fa4319-1945-458f-bd21-05334c51cbb4",
            item_id="ed3aad28-de5d-43d5-8a97-a6988901c921",
            name="dev_source",
            kind="lakehouse",
            schema_enabled=LAKEHOUSE_SCHEMAS_ENABLED,
            schema=SOURCE_SCHEMA,
        ),
        "unified": FabricStore(
            env=ENV,
            workspace_id="68fa4319-1945-458f-bd21-05334c51cbb4",
            item_id="107d4b73-7c0e-4ce1-8da8-7602ac4a1372",
            name="dev_unified",
            kind="lakehouse",
            schema_enabled=LAKEHOUSE_SCHEMAS_ENABLED,
            schema=UNIFIED_SCHEMA,
        ),
        "product": FabricStore(
            env=ENV,
            workspace_id="68fa4319-1945-458f-bd21-05334c51cbb4",
            item_id="185346e5-6ce4-40c8-852a-185f1a933d20",
            name="dev_product",
            kind="warehouse",
            schema_enabled=LAKEHOUSE_SCHEMAS_ENABLED,
            schema=PRODUCT_SCHEMA,
        ),
        "metadata": FabricStore(
            env=ENV,
            workspace_id="68fa4319-1945-458f-bd21-05334c51cbb4",
            item_id="329b3989-4546-4331-b9ff-df898d49ee73",
            name="gov_metadata",
            kind="lakehouse",
            schema_enabled=LAKEHOUSE_SCHEMAS_ENABLED,
            schema=METADATA_SCHEMA,
        ),
    }
}

PATH_CONFIG = PathConfig(paths=ENV_PATHS)


### 01_agreement Metadata Intake Config

The two `01_agreement` widgets expose only lightweight business fields. Add organization-specific fields here; the widgets store those values in `custom_fields_json` without changing package code or table schemas.


In [ ]:
DATA_AGREEMENT_CONFIG = DataAgreementConfig(
    metadata_tables={
        "data_steward": "METADATA_DATA_STEWARD",
        "data_agreement": "METADATA_DATA_AGREEMENT",
        "data_agreement_evidence": "METADATA_DATA_AGREEMENT_EVIDENCE",
    },
    steward_role_options=[
        "Data Owner",
        "Data Steward",
        "Data Custodian",
        "Governance Reviewer",
        "Business Approver",
    ],
    data_steward_widget={
        "visible_columns": [
            "steward_name", "steward_role", "contact", "effective_from", "effective_to",
        ],
        "custom_fields": [
            {
                "key": "optional",
                "label": "Optional",
                "type": "text",
                "required": False,
                "help": "Optional group of users or organization unit covered by the agreement.",
            },
            {
                "key": "optional_dropdown",
                "label": "Optional dropdown",
                "type": "select",
                "required": False,
                "options": ["ODI", "Faculty", "Department", "Research Group", "Other"],
            },
        ],
    },
    data_agreement_widget={
        "visible_columns": [
            "agreement_name", "domain", "steward_id", "recipient", "start_date",
            "expiry_date", "business_purpose", "approved_usage_internal",
            "approved_usage_external", "approved_usage_research",
        ],
        "custom_fields": [
            {
                "key": "optional",
                "label": "Optional",
                "type": "text",
                "required": False,
                "help": "Optional group of users or organization unit covered by the agreement.",
            },
            {
                "key": "optional_dropdown",
                "label": "Optional dropdown",
                "type": "select",
                "required": False,
                "options": ["ODI", "Faculty", "Department", "Research Group", "Other"],
            },
        ],
    },
)


Creates or checks the current 11 FabricOps metadata tables in `CONFIG.path_config.paths[ENV]["metadata"]`. Metadata setup creates any missing tables in the configured metadata lakehouse target through the FabricOps lakehouse write helper; it does not require the notebook to have a default lakehouse attached. Older or malformed metadata tables are not automatically migrated; recreate or manually migrate tables if schema validation reports missing columns.


### AI prompt config

FabricOps keeps the implemented default prompt templates in package code so this user-facing notebook stays compact. Override `AIPromptConfig(...)` only when your project has reviewed replacement prompt text.


In [ ]:
AI_PROMPTS = AIPromptConfig()


## Config Compiler & Bootstrap

In [ ]:
CONFIG = FrameworkConfig(
    path_config=PATH_CONFIG,
    notebook_runtime_config=RUNTIME_CONFIG,
    ai_prompt_config=AI_PROMPTS,
    data_agreement_config=DATA_AGREEMENT_CONFIG,
    audit_timezone=FABRICOPS_AUDIT_TIMEZONE,
)

RUN_CONTEXT = setup_notebook(
    config=CONFIG,
    env=ENV,
    required_targets=REQUIRED_TARGETS,
)



In [ ]:
# Optional one-time metadata table setup.
#
# Run this block manually once per environment.
# Rerun it only after metadata schema/table changes.
# Keep it commented during normal downstream notebook runs because
# 01_agreement, 02_pipeline, and 03_governance use `%run 00_env_config`.
#
# To run setup, uncomment the block below, execute this cell once, then
# comment it back before normal use.
#
# METADATA_TABLE_SETUP = setup_metadata_tables(
#     spark=spark,
#     config=CONFIG,
#     env=ENV,
#     metadata_schema=METADATA_SCHEMA,
#     require_active_steward=False,
# )
# AGREEMENT_METADATA_SETUP = METADATA_TABLE_SETUP["data_agreement"]
# print("Metadata table setup:", METADATA_TABLE_SETUP)


In [ ]:
print("FabricOps environment bootstrap ready")
print(f"- env: {ENV}")
print(f"- validation mode: {VALIDATION_MODE}")
print(f"- source target: {CONFIG.path_config.paths[ENV]['source'].name}")
print(f"- unified target: {CONFIG.path_config.paths[ENV]['unified'].name}")
print(f"- product target: {CONFIG.path_config.paths[ENV]['product'].name}")
print(f"- metadata target: {CONFIG.path_config.paths[ENV]['metadata'].name}")

metadata_table_setup = globals().get("METADATA_TABLE_SETUP")
agreement_metadata_setup = globals().get("AGREEMENT_METADATA_SETUP")

if metadata_table_setup is None:
    print("- metadata table setup: skipped")
else:
    print(f"- metadata table setup: {metadata_table_setup}")

if agreement_metadata_setup is not None:
    print(f"- 01_agreement metadata tables created/checked: {agreement_metadata_setup['tables']}")

print(f"- 01_agreement widget table names: {CONFIG.data_agreement_config.metadata_tables}")
print(f"- data steward custom fields: {CONFIG.data_agreement_config.data_steward_widget['custom_fields']}")
print(f"- data agreement custom fields: {CONFIG.data_agreement_config.data_agreement_widget['custom_fields']}")

if agreement_metadata_setup is not None:
    print(f"- steward readiness: {agreement_metadata_setup['status']}")
    print(f"  - message: {agreement_metadata_setup['message']}")
    print(f"  - active steward count: {agreement_metadata_setup['active_steward_count']}")
    if VALIDATION_MODE == "strict" and agreement_metadata_setup["status"] != "ready":
        raise RuntimeError(agreement_metadata_setup["message"])

print(f"- audit timezone: {CONFIG.audit_timezone}")

if metadata_table_setup is None:
    print("- metadata table validation: skipped")
else:
    metadata_registration_validation = metadata_table_setup["registration_validation"]
    print(f"- active metadata table count: {metadata_table_setup['active_metadata_table_count']}")
    if metadata_registration_validation["status"] == "ready":
        print("Metadata table validation passed.")
    else:
        print("Metadata table validation warning or failure.")
    print("Configured metadata tables validated:")
    for table_name in metadata_registration_validation.get("registered_tables", []):
        print(f"  - {table_name}")
    if metadata_registration_validation.get("missing_tables"):
        print("Missing configured metadata tables:")
        for table_name in metadata_registration_validation["missing_tables"]:
            print(f"  - {table_name}")
    for warning in metadata_registration_validation.get("warnings", []):
        print(f"Metadata warning: {warning}")

    if VALIDATION_MODE == "strict" and metadata_registration_validation["status"] not in {"ready", "skipped"}:
        raise RuntimeError(
            "Metadata table validation failed. Missing: "
            + ", ".join(metadata_registration_validation["missing_tables"])
        )
